# Pitcher appearances

Inspect and validate the reusable pitcher-appearance view derived from local Statcast pitch records. The view has one row per pitcher, pitching team, and game; starts and relief appearances are classified at the appearance level.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

from crooked_numbers_analytics.appearances import (
    create_pitcher_appearances_view,
)
from crooked_numbers_analytics.db import open_database

con = open_database()

## 2. Research parameters

Set `seasons = None` to use every locally available season. Change `team` for manual inspection without rebuilding the view.

In [ ]:
seasons = [2023, 2024]
team = "MIL"

## 3. Create and inspect the view

The helper reads the selected Hive partitions directly and defaults to regular-season games (`game_type = 'R'`). It does not materialize another dataset.

In [ ]:
create_pitcher_appearances_view(con, seasons=seasons)

In [ ]:
con.execute(
    """
    SELECT *
    FROM pitcher_appearances
    WHERE pitching_team = ?
    ORDER BY game_date, game_pk, is_start DESC, first_inning, pitcher_id
    LIMIT 100
    """,
    [team],
).df()

## 4. Coverage and sanity checks

`pitch_count` is the number of Statcast pitch records in an appearance. Compare edge cases with official pitch counts before treating the values as perfectly equivalent.

In [ ]:
con.execute(
    """
    SELECT
        season,
        COUNT(DISTINCT game_pk) AS games,
        COUNT(*) AS appearances,
        COUNT(*) FILTER (WHERE is_start) AS starts,
        COUNT(*) FILTER (WHERE NOT is_start) AS relief_appearances,
        MIN(pitch_count) AS min_pitch_count,
        MAX(pitch_count) AS max_pitch_count
    FROM pitcher_appearances
    WHERE pitching_team = ?
    GROUP BY season
    ORDER BY season
    """,
    [team],
).df()

In [ ]:
con.sql(
    """
    SELECT game_pk, pitching_team, pitcher_id, COUNT(*) AS rows
    FROM pitcher_appearances
    GROUP BY game_pk, pitching_team, pitcher_id
    HAVING COUNT(*) > 1
    ORDER BY game_pk, pitching_team, pitcher_id
    """
).df()

In [ ]:
con.sql(
    """
    WITH starters AS (
        SELECT
            game_pk, pitching_team,
            COUNT(*) FILTER (WHERE is_start) AS starter_count
        FROM pitcher_appearances
        GROUP BY game_pk, pitching_team
    )
    SELECT *
    FROM starters
    WHERE starter_count <> 1
    ORDER BY game_pk, pitching_team
    """
).df()

In [ ]:
con.sql(
    """
    SELECT *
    FROM pitcher_appearances
    WHERE pitcher_id IS NULL
       OR pitching_team IS NULL
       OR pitch_count = 0
    ORDER BY game_date, game_pk
    """
).df()

## 5. Manual game inspection

Choose a game from the team sample and inspect both clubs, with starts first and inning spans visible.

In [ ]:
inspection_game_pk = con.execute(
    """
    SELECT game_pk
    FROM pitcher_appearances
    WHERE pitching_team = ?
    ORDER BY game_date, game_pk
    LIMIT 1
    """,
    [team],
).fetchone()[0]

con.execute(
    """
    SELECT *
    FROM pitcher_appearances
    WHERE game_pk = ?
    ORDER BY pitching_team, is_start DESC, first_inning, pitcher_id
    """,
    [inspection_game_pk],
).df()